# Quality check

What I need to check is:
- every selected tree is taken care of. Possible states are:
    - Alive
    - dead
    - no a tree
- QGIS table and exported files are consistent 
- folder structure
- reference system exported point clouds

![Manual segmentation flowchart](segmentation-workflow.png)

This workflow involes a lot of manual steps where things can go wrong, they include:

- trees that RiSCAN Pro doesn't see but exist on disk
- new trees that were not added to QGIS
- trees that got removed because they are not tree like lianas
- trees that are dead non marked as dead
- genuinily lost files

the idea of this notebook is that you look at the errors and iteratively fix things

# Check tree consistency

In [ ]:
project_name = "{{ project_name }}"

In [52]:
from pyprojroot import here

crowns_path = here(f"{project_name}_crowns.gpkg")
dir_cor = here("Corrected_pcs")
dir_sel = here("Propagation/selected_trees")

In [53]:
import polars as pl
from polars import col as c
from fastcore.all import *
import geopandas as gpd

load selected trees into a df

In [54]:
id_re = r"_(\d{6}_\d{7}_\d{2}[a-z]?)"

In [55]:
sel_df = pl.DataFrame({"sel_name": [f.stem for f in dir_sel.ls()]}).with_columns(
    sel_id=c.sel_name.str.extract(id_re, 1)
)
sel_df

sel_name,sel_id
str,str
"""FIS01-02_345910_8059490_02""","""345910_8059490_02"""
"""FIS01-02_345910_8059490_05""","""345910_8059490_05"""
"""FIS01-02_345910_8059490_06""","""345910_8059490_06"""
"""FIS01-02_345910_8059490_07""","""345910_8059490_07"""
"""FIS01-02_345910_8059490_09""","""345910_8059490_09"""
…,…
"""FIS01-02_345970_8059550_51""","""345970_8059550_51"""
"""FIS01-02_345980_8059550_01""","""345980_8059550_01"""
"""FIS01-02_345980_8059550_06d""","""345980_8059550_06d"""


In [56]:
cor_df = pl.DataFrame({"cor_name": [f.stem for f in dir_cor.ls()]}).with_columns(
    cor_id=c.cor_name.str.extract(id_re, 1)
)
cor_df

cor_name,cor_id
str,str
"""FIS01-02_345910_8059490_02_cor""","""345910_8059490_02"""
"""FIS01-02_345910_8059490_05_cor""","""345910_8059490_05"""
"""FIS01-02_345910_8059490_06_cor""","""345910_8059490_06"""
"""FIS01-02_345910_8059490_07_cor""","""345910_8059490_07"""
"""FIS01-02_345910_8059490_09_cor""","""345910_8059490_09"""
…,…
"""FIS01-02_345970_8059550_28_cor""","""345970_8059550_28"""
"""FIS01-02_345970_8059550_34_cor""","""345970_8059550_34"""
"""FIS01-02_345970_8059550_51_cor""","""345970_8059550_51"""


Note a tree can be both dead and new (why would I do that?)

In [57]:
crowns_gdf = gpd.read_file(crowns_path, layer=crowns_path.stem)
crowns_df = (
    pl.from_pandas(crowns_gdf.drop(columns="geometry"))
    .with_columns(
        c.segmented.cast(pl.Boolean),
        c.is_dead.cast(pl.Boolean),
        c.is_removed.cast(pl.Boolean),
        c.is_new.cast(pl.Boolean),
    )
    .with_columns(
        state=pl.when(c.is_dead)
        .then(pl.lit("dead"))
        .when(c.is_removed)
        .then(pl.lit("removed"))
        .when(c.is_new)
        .then(pl.lit("new"))
        .when(c.segmented)
        .then(pl.lit("segmented"))
    )
    .with_columns(c.state.cast(pl.Categorical))
)
crowns_df

GIS_ID,ID_number,area,area.1,cell_id,grid_x,grid_y,grid_x_norm,grid_y_norm,hilbert_dist,suffix,pretty_name,segmented,notes,is_dead,is_removed,is_new,state
str,str,f64,f64,str,f64,f64,f64,f64,f64,str,str,bool,str,bool,bool,bool,cat
"""FIS01-02_345910_8059490_02""","""345910_8059490_02""",3.651367,3.651367,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""02""","""Lama_02""",true,null,false,null,null,"""segmented"""
"""FIS01-02_345910_8059490_05""","""345910_8059490_05""",16.01758,16.01758,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""05""","""Lama_05""",true,null,false,null,null,"""segmented"""
"""FIS01-02_345910_8059490_06""","""345910_8059490_06""",4.727412,4.727412,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""06""","""Lama_06""",true,null,false,null,null,"""segmented"""
"""FIS01-02_345910_8059490_07""","""345910_8059490_07""",10.067097,10.067097,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""07""","""Lama_07""",true,null,false,null,null,"""segmented"""
"""FIS01-02_345910_8059490_09""","""345910_8059490_09""",3.543199,3.543199,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""09""","""Lama_09""",true,null,false,null,null,"""segmented"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FIS01-02_345910_8059500_27""","""345910_8059500_27""",4.548914,4.548914,"""345910_8059500""",345910.0,8.0595e6,0.0,30.0,1020.0,"""23""","""Mada_27""",true,"""dead""",true,null,true,"""dead"""
"""FIS01-02_345920_8059500_28""","""345920_8059500_28""",3.454107,3.454107,"""345920_8059500""",345920.0,8.0595e6,10.0,30.0,786.0,"""25""","""Jaya_28""",true,null,false,null,true,"""new"""
"""FIS01-02_345940_8059540_04""","""345940_8059540_04""",34.558191,34.558191,"""345940_8059540""",345940.0,8.05954e6,30.0,70.0,14658.0,"""01""","""Zora_04""",true,null,false,null,true,"""new"""


there are no empty states

In [58]:
assert crowns_df.select(c.state.is_not_null().all()).item()

an empty state can only mean that a tree is selected but no in the crowns

In [59]:
joined_crowns = (
    sel_df.join(crowns_df, left_on="sel_id", right_on="ID_number", how="full")
    .with_columns(
        c.state.fill_null("sel_missing_in_crowns")
    )
    .join(cor_df, left_on="ID_number", right_on="cor_id", how="full")
)
joined_crowns

sel_name,sel_id,GIS_ID,ID_number,area,area.1,cell_id,grid_x,grid_y,grid_x_norm,grid_y_norm,hilbert_dist,suffix,pretty_name,segmented,notes,is_dead,is_removed,is_new,state,cor_name,cor_id
str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,str,str,bool,str,bool,bool,bool,cat,str,str
"""FIS01-02_345910_8059490_02""","""345910_8059490_02""","""FIS01-02_345910_8059490_02""","""345910_8059490_02""",3.651367,3.651367,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""02""","""Lama_02""",true,null,false,null,null,"""segmented""","""FIS01-02_345910_8059490_02_cor""","""345910_8059490_02"""
"""FIS01-02_345910_8059490_05""","""345910_8059490_05""","""FIS01-02_345910_8059490_05""","""345910_8059490_05""",16.01758,16.01758,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""05""","""Lama_05""",true,null,false,null,null,"""segmented""","""FIS01-02_345910_8059490_05_cor""","""345910_8059490_05"""
"""FIS01-02_345910_8059490_06""","""345910_8059490_06""","""FIS01-02_345910_8059490_06""","""345910_8059490_06""",4.727412,4.727412,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""06""","""Lama_06""",true,null,false,null,null,"""segmented""","""FIS01-02_345910_8059490_06_cor""","""345910_8059490_06"""
"""FIS01-02_345910_8059490_07""","""345910_8059490_07""","""FIS01-02_345910_8059490_07""","""345910_8059490_07""",10.067097,10.067097,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""07""","""Lama_07""",true,null,false,null,null,"""segmented""","""FIS01-02_345910_8059490_07_cor""","""345910_8059490_07"""
"""FIS01-02_345910_8059490_09""","""345910_8059490_09""","""FIS01-02_345910_8059490_09""","""345910_8059490_09""",3.543199,3.543199,"""345910_8059490""",345910.0,8.05949e6,0.0,20.0,944.0,"""09""","""Lama_09""",true,null,false,null,null,"""segmented""","""FIS01-02_345910_8059490_09_cor""","""345910_8059490_09"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,null,"""FIS01-02_345920_8059500_28""","""345920_8059500_28""",3.454107,3.454107,"""345920_8059500""",345920.0,8.0595e6,10.0,30.0,786.0,"""25""","""Jaya_28""",true,null,false,null,true,"""new""","""FIS01-02_345920_8059500_28_cor""","""345920_8059500_28"""
null,null,"""FIS01-02_345940_8059540_04""","""345940_8059540_04""",34.558191,34.558191,"""345940_8059540""",345940.0,8.05954e6,30.0,70.0,14658.0,"""01""","""Zora_04""",true,null,false,null,true,"""new""","""FIS01-02_345940_8059540_04_cor""","""345940_8059540_04"""
null,null,"""FIS01-02_345930_8059520_18""","""345930_8059520_18""",null,null,null,null,null,null,null,null,null,"""Okea_18""",true,null,false,null,true,"""new""","""FIS01-02_345930_8059520_18_cor""","""345930_8059520_18"""


In [60]:
joined_crowns.group_by("state").agg(pl.len())

state,len
cat,u32
"""removed""",1
null,1
"""dead""",43
"""new""",5
"""segmented""",461


Those assertions should be true


| state | in selection | in corrected |
|-----------|:---:|:---:|
| removed | ✅ | ❌ |
| new | ❌ | ✅ |
| dead | ✅ | ❌ |
| segmented | ✅ | ✅ | 

In [61]:
removed = joined_crowns.filter(c.state == "removed")
assert removed.select(c.sel_name.is_not_null().all()).item(), "removed trees should all be in selection"
assert removed.select(c.cor_name.is_null().all()).item(), "removed trees should not be in corrected"

new = joined_crowns.filter(c.state == "new")
assert new.select(c.sel_name.is_null().all()).item(), "new trees should not be in selection"
assert new.select(c.cor_name.is_not_null().all()).item(), "new trees should all be in corrected"

dead = joined_crowns.filter(c.state == "dead", ~c.is_new)
assert dead.select(c.sel_name.is_not_null().all()).item(), "dead trees should all be in selection"
assert dead.select(c.cor_name.is_null().all()).item(), "dead trees should not be in corrected"

segmented = joined_crowns.filter(c.state == "segmented")
assert segmented.select(c.sel_name.is_not_null().all()).item(), "segmented trees should all be in selection"
assert segmented.select(c.cor_name.is_not_null().all()).item(), "segmented trees should all be in corrected"


Ensure that the tree IDs are unique

In [ ]:
assert joined_crowns.drop_nulls("cor_id").group_by(c.cor_id).agg(pl.len() == 1).select(c.len.all()).item(), "all corrected trees should be unique"
assert joined_crowns.drop_nulls("sel_id").group_by(c.sel_id).agg(pl.len() == 1).select(c.len.all()).item(), "all selected trees should be unique"
assert joined_crowns.drop_nulls("ID_number").group_by(c.ID_number).agg(pl.len() == 1).select(c.len.all()).item(), "all selected trees should be unique"

# Check folder structure

In [66]:
tree_regex = r"\w{3}\d{2}-\d{2}_\d{6}_\d{7}_\d{2}[a-z]?"

In [67]:
required_folders = {
    "Corrected_pcs": tree_regex+"_cor",
    "Propagation/selected_trees": tree_regex
}

required_files = [
    f'{project_name}_crowns.gpkg',
    f'{project_name}_qgis.qgz',
    f'{project_name}_qgis_table.xlsx',
]

In [68]:
import re

for folder, pattern in required_folders.items():
    folder_path = here(folder)
    assert folder_path.exists(), f"missing folder: {folder}"
    unexpected = [f.name for f in folder_path.ls() if not re.fullmatch(pattern, f.stem)]
    assert not unexpected, f"unexpected files in {folder}:\n" + "\n".join(unexpected)

for fname in required_files:
    assert here(fname).exists(), f"missing file: {fname}"


# Check reference system

In [74]:
import laspy
def get_crs(cloud: Path):
    with laspy.open(cloud) as f:
        return f.header.parse_crs()


In [75]:
test_cloud = dir_cor.ls()[0]
crs = get_crs(test_cloud)

In [80]:
crs

<Projected CRS: COMPD_CS["WGS 84 / UTM zone 55S",PROJCS["WGS 84 /  ...>
Name: WGS 84 / UTM zone 55S
Axis Info [cartesian]:
- [east]: Easting (Meter)
- [north]: Northing (Meter)
- h[up]: Ellipsoidal height (Meter)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Transverse Mercator
Datum: WGS84
- Ellipsoid: WGS84
- Prime Meridian: Greenwich

In [85]:
assert all(dir_cor.ls().map(get_crs).map(lambda x: x==crs)), "not all point clouds have the same CRS"